# Algorytmy 2PC i 3PC

W tym notatniku przyjrzymy się dwóm protokołom konsensusu stosowanym w rozproszonych systemach transakcyjnych: **dwufazowemu zatwierdzaniu (2PC)** i **trójfazowemu zatwierdzaniu (3PC)**. Zaimplementujemy symulacje obu algorytmów, aby zrozumieć ich działanie, zalety i wady.

## Dwufazowe zatwierdzanie (2PC - Two-Phase Commit)

### Faza 1: Faza głosowania (Voting Phase)

1.  **Koordynator** wysyła do wszystkich **uczestników** (nazywanych też pracownikami lub kohortami) wiadomość `VOTE_REQUEST`.
2.  Każdy **uczestnik**, który jest gotowy do zatwierdzenia transakcji, zapisuje swój stan w stabilnej pamięci i wysyła do koordynatora wiadomość `VOTE_COMMIT`. W przeciwnym razie wysyła `VOTE_ABORT`.

### Faza 2: Faza zatwierdzania (Commit Phase)

1.  **Koordynator** zbiera głosy od wszystkich uczestników:
    *   Jeśli wszyscy uczestnicy zagłosowali `VOTE_COMMIT`, koordynator wysyła do wszystkich wiadomość `GLOBAL_COMMIT`.
    *   Jeśli którykolwiek z uczestników zagłosował `VOTE_ABORT` (lub nie odpowiedział w wyznaczonym czasie), koordynator wysyła do wszystkich wiadomość `GLOBAL_ABORT`.
2.  **Uczestnicy** na podstawie otrzymanej wiadomości od koordynatora zatwierdzają (`COMMIT`) lub przerywają (`ABORT`) transakcję i zwalniają zasoby.

### Diagram przepływu 2PC

```mermaid
sequenceDiagram
    participant Koordynator
    participant Uczestnik 1
    participant Uczestnik 2

    Koordynator->>Uczestnik 1: VOTE_REQUEST
    Koordynator->>Uczestnik 2: VOTE_REQUEST
    
    Uczestnik 1-->>Koordynator: VOTE_COMMIT
    Uczestnik 2-->>Koordynator: VOTE_COMMIT
    
    Koordynator->>Uczestnik 1: GLOBAL_COMMIT
    Koordynator->>Uczestnik 2: GLOBAL_COMMIT
```

### Implementacja symulacji 2PC

In [3]:
import time
import random

class Participant:
    def __init__(self, name, should_commit):
        self.name = name
        self.should_commit = should_commit
        self.state = 'IDLE'

    def vote(self):
        if self.should_commit:
            self.state = 'READY'
            print(f"{self.name}: Głosuję VOTE_COMMIT")
            return 'VOTE_COMMIT'
        else:
            self.state = 'ABORTED'
            print(f"{self.name}: Głosuję VOTE_ABORT")
            return 'VOTE_ABORT'

    def final_decision(self, decision):
        if decision == 'GLOBAL_COMMIT':
            self.state = 'COMMITTED'
            print(f"{self.name}: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.")
        else:
            self.state = 'ABORTED'
            print(f"{self.name}: Otrzymano GLOBAL_ABORT. Przerywam transakcję.")

class Coordinator:
    def __init__(self, participants):
        self.participants = participants
        self.state = 'IDLE'

    def run_transaction(self):
        print("--- Faza 1: Głosowanie ---")
        self.state = 'WAITING'
        votes = []
        for p in self.participants:
            votes.append(p.vote())

        print("\n--- Faza 2: Decyzja ---")
        if all(v == 'VOTE_COMMIT' for v in votes):
            print("Koordynator: Wszyscy uczestnicy zgodzili się. Wysyłam GLOBAL_COMMIT.")
            self.state = 'COMMITTED'
            for p in self.participants:
                p.final_decision('GLOBAL_COMMIT')
        else:
            print("Koordynator: Nie wszyscy uczestnicy się zgodzili. Wysyłam GLOBAL_ABORT.")
            self.state = 'ABORTED'
            for p in self.participants:
                p.final_decision('GLOBAL_ABORT')

print("### Scenariusz 1: Wszyscy uczestnicy zatwierdzają transakcję ###")
participants_commit = [Participant(f"Uczestnik-{i}", True) for i in range(1, 4)]
coordinator_commit = Coordinator(participants_commit)
coordinator_commit.run_transaction()

print("\n### Scenariusz 2: Jeden z uczestników przerywa transakcję ###")
participants_abort = [
    Participant("Uczestnik-1", True),
    Participant("Uczestnik-2", False), # Ten uczestnik przerwie
    Participant("Uczestnik-3", True)
]
coordinator_abort = Coordinator(participants_abort)
coordinator_abort.run_transaction()

### Scenariusz 1: Wszyscy uczestnicy zatwierdzają transakcję ###
--- Faza 1: Głosowanie ---
Uczestnik-1: Głosuję VOTE_COMMIT
Uczestnik-2: Głosuję VOTE_COMMIT
Uczestnik-3: Głosuję VOTE_COMMIT

--- Faza 2: Decyzja ---
Koordynator: Wszyscy uczestnicy zgodzili się. Wysyłam GLOBAL_COMMIT.
Uczestnik-1: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.
Uczestnik-2: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.
Uczestnik-3: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.

### Scenariusz 2: Jeden z uczestników przerywa transakcję ###
--- Faza 1: Głosowanie ---
Uczestnik-1: Głosuję VOTE_COMMIT
Uczestnik-2: Głosuję VOTE_ABORT
Uczestnik-3: Głosuję VOTE_COMMIT

--- Faza 2: Decyzja ---
Koordynator: Nie wszyscy uczestnicy się zgodzili. Wysyłam GLOBAL_ABORT.
Uczestnik-1: Otrzymano GLOBAL_ABORT. Przerywam transakcję.
Uczestnik-2: Otrzymano GLOBAL_ABORT. Przerywam transakcję.
Uczestnik-3: Otrzymano GLOBAL_ABORT. Przerywam transakcję.


### Problem blokowania w 2PC

Główną wadą 2PC jest to, że jest to protokół **blokujący**. Jeśli koordynator ulegnie awarii po rozpoczęciu fazy zatwierdzania, a przed wysłaniem ostatecznej decyzji, uczestnicy, którzy zagłosowali `VOTE_COMMIT` pozostają w stanie niepewności (`READY`). Nie wiedzą, czy transakcja została zatwierdzona czy przerwana. Muszą czekać, aż koordynator wróci do działania, blokując przy tym zasoby.

## Trójfazowe zatwierdzanie (3PC - Three-Phase Commit)

3PC został zaprojektowany w celu rozwiązania problemu blokowania w 2PC. Dodaje on dodatkową fazę, która pozwala systemowi kontynuować działanie nawet w przypadku awarii koordynatora.

### Faza 1: Faza głosowania (Voting Phase)
Podobna do 2PC. Koordynator wysyła `VOTE_REQUEST`, a uczestnicy odpowiadają `VOTE_COMMIT` lub `VOTE_ABORT`.

### Faza 2: Faza przygotowania do zatwierdzenia (Prepare-to-Commit Phase)

1.  Jeśli koordynator otrzymał `VOTE_COMMIT` od wszystkich uczestników, wysyła do nich wiadomość `PREPARE_TO_COMMIT`.
2.  Jeśli otrzymał jakikolwiek `VOTE_ABORT`, wysyła `GLOBAL_ABORT`.
3.  Uczestnicy po otrzymaniu `PREPARE_TO_COMMIT` przechodzą w stan `PREPARED` i potwierdzają to koordynatorowi.

### Faza 3: Faza zatwierdzania (Commit Phase)

1.  Gdy koordynator otrzyma potwierdzenia od wszystkich uczestników, że są w stanie `PREPARED`, wysyła `GLOBAL_COMMIT`.
2.  Uczestnicy zatwierdzają transakcję.

### Diagram przepływu 3PC

```mermaid
sequenceDiagram
    participant Koordynator
    participant Uczestnik 1
    participant Uczestnik 2

    Koordynator->>Uczestnik 1: VOTE_REQUEST
    Koordynator->>Uczestnik 2: VOTE_REQUEST
    
    Uczestnik 1-->>Koordynator: VOTE_COMMIT
    Uczestnik 2-->>Koordynator: VOTE_COMMIT
    
    Koordynator->>Uczestnik 1: PREPARE_TO_COMMIT
    Koordynator->>Uczestnik 2: PREPARE_TO_COMMIT
    
    Uczestnik 1-->>Koordynator: ACK
    Uczestnik 2-->>Koordynator: ACK
    
    Koordynator->>Uczestnik 1: GLOBAL_COMMIT
    Koordynator->>Uczestnik 2: GLOBAL_COMMIT
```

### Implementacja symulacji 3PC

In [5]:
import time
import random

class Participant3PC:
    def __init__(self, name, should_commit):
        self.name = name
        self.should_commit = should_commit
        self.state = 'IDLE'

    def vote(self):
        if self.should_commit:
            self.state = 'READY'
            print(f"{self.name}: Głosuję VOTE_COMMIT")
            return 'VOTE_COMMIT'
        else:
            self.state = 'ABORTED'
            print(f"{self.name}: Głosuję VOTE_ABORT")
            return 'VOTE_ABORT'
            
    def prepare(self):
        self.state = 'PREPARED'
        print(f"{self.name}: Otrzymano PREPARE_TO_COMMIT. Potwierdzam.")
        return 'ACK'

    def final_decision(self, decision):
        if decision == 'GLOBAL_COMMIT':
            self.state = 'COMMITTED'
            print(f"{self.name}: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.")
        else:
            self.state = 'ABORTED'
            print(f"{self.name}: Otrzymano GLOBAL_ABORT. Przerywam transakcję.")

class Coordinator3PC:
    def __init__(self, participants):
        self.participants = participants
        self.state = 'IDLE'

    def run_transaction(self, simulate_coordinator_failure=False):
        print("--- Faza 1: Głosowanie ---")
        self.state = 'WAITING'
        votes = []
        for p in self.participants:
            votes.append(p.vote())

        if all(v == 'VOTE_COMMIT' for v in votes):
            print("\n--- Faza 2: Przygotowanie do zatwierdzenia ---")
            self.state = 'PREPARING'
            acks = []
            for p in self.participants:
                acks.append(p.prepare())
            
            if simulate_coordinator_failure:
                print("\n!!! Symulacja awarii koordynatora po fazie przygotowania !!!")
                print("Uczestnicy mogą bezpiecznie zatwierdzić transakcję po upływie timeoutu, ponieważ wszyscy byli w stanie PREPARED.")
                return # Zakończ symulację awarii
            
            if all(a == 'ACK' for a in acks):
                print("\n--- Faza 3: Zatwierdzenie ---")
                print("Koordynator: Wszyscy uczestnicy gotowi. Wysyłam GLOBAL_COMMIT.")
                self.state = 'COMMITTED'
                for p in self.participants:
                    p.final_decision('GLOBAL_COMMIT')
        else:
            print("\n--- Decyzja: Przerwanie ---")
            print("Koordynator: Nie wszyscy uczestnicy się zgodzili. Wysyłam GLOBAL_ABORT.")
            self.state = 'ABORTED'
            for p in self.participants:
                p.final_decision('GLOBAL_ABORT')

print("### Scenariusz 3: Transakcja 3PC zakończona sukcesem ###")
participants_3pc_commit = [Participant3PC(f"Uczestnik-{i}", True) for i in range(1, 4)]
coordinator_3pc_commit = Coordinator3PC(participants_3pc_commit)
coordinator_3pc_commit.run_transaction()

print("\n### Scenariusz 4: Symulacja awarii koordynatora w 3PC ###")
participants_3pc_failure = [Participant3PC(f"Uczestnik-{i}", True) for i in range(1, 4)]
coordinator_3pc_failure = Coordinator3PC(participants_3pc_failure)
coordinator_3pc_failure.run_transaction(simulate_coordinator_failure=True)

### Scenariusz 3: Transakcja 3PC zakończona sukcesem ###
--- Faza 1: Głosowanie ---
Uczestnik-1: Głosuję VOTE_COMMIT
Uczestnik-2: Głosuję VOTE_COMMIT
Uczestnik-3: Głosuję VOTE_COMMIT

--- Faza 2: Przygotowanie do zatwierdzenia ---
Uczestnik-1: Otrzymano PREPARE_TO_COMMIT. Potwierdzam.
Uczestnik-2: Otrzymano PREPARE_TO_COMMIT. Potwierdzam.
Uczestnik-3: Otrzymano PREPARE_TO_COMMIT. Potwierdzam.

--- Faza 3: Zatwierdzenie ---
Koordynator: Wszyscy uczestnicy gotowi. Wysyłam GLOBAL_COMMIT.
Uczestnik-1: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.
Uczestnik-2: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.
Uczestnik-3: Otrzymano GLOBAL_COMMIT. Zatwierdzam transakcję.

### Scenariusz 4: Symulacja awarii koordynatora w 3PC ###
--- Faza 1: Głosowanie ---
Uczestnik-1: Głosuję VOTE_COMMIT
Uczestnik-2: Głosuję VOTE_COMMIT
Uczestnik-3: Głosuję VOTE_COMMIT

--- Faza 2: Przygotowanie do zatwierdzenia ---
Uczestnik-1: Otrzymano PREPARE_TO_COMMIT. Potwierdzam.
Uczestnik-2: Otrzymano PREPARE_TO_CO

### Jak 3PC rozwiązuje problem blokowania?

Stan `PREPARED` jest kluczowy. Jeśli koordynator ulegnie awarii, a uczestnicy są w stanie `PREPARED`, wiedzą, że wszyscy inni uczestnicy również musieli zagłosować za zatwierdzeniem. Po upływie określonego czasu (timeout) mogą bezpiecznie założyć, że transakcja zostanie zatwierdzona i dokonać `COMMIT`. Gdyby którykolwiek z uczestników zagłosował `VOTE_ABORT`, koordynator nigdy nie wysłałby `PREPARE_TO_COMMIT`, więc żaden uczestnik nie osiągnąłby stanu `PREPARED`.

## Porównanie 2PC i 3PC

| Cecha | 2PC (Dwufazowe zatwierdzanie) | 3PC (Trójfazowe zatwierdzanie) |
|---|---|---|
| **Problem blokowania** | **Tak**, jest to protokół blokujący. Awaria koordynatora może zablokować zasoby. | **Nie**, jest to protokół nieblokujący. | 
| **Złożoność** | Prostszy, mniejsza liczba komunikatów. | Bardziej złożony, wymaga dodatkowej fazy i komunikatów. |
| **Wydajność** | Wyższa wydajność i mniejsze opóźnienia w przypadku braku awarii. | Niższa wydajność i większe opóźnienia z powodu dodatkowej fazy. |
| **Odporność na awarie** | Wrażliwy na awarię koordynatora. | Bardziej odporny na awarię koordynatora. |

## Podsumowanie

2PC jest lżejszy i szybszy, ale jego blokująca natura czyni go ryzykownym w systemach, gdzie wysoka dostępność jest krytyczna. 3PC, mimo że jest bardziej skomplikowany i ma większy narzut komunikacyjny, zapewnia większą odporność na awarie, eliminując problem blokowania, co jest kluczowe w wielu systemach rozproszonych o wysokich wymaganiach.